# Exploration: collecting the BNF and BNF-EN metadata

BNF exposes its metadata through a wide range of API formats. 
The goal here is to explore the various options to get an idea of which would be most appropriate for our needs.

An API with records in a marc related format would be ideal, since it's what the importer expects.
However, it is desireable to compare what information is avalaible through other requests to ensure all useful information possible is gathered.

### Imports

In [ ]:
import os
import json
import requests
from bs4 import BeautifulSoup
from impresso_db.models.enums import EnumMetadataProp
from ast import literal_eval

from impresso_commons.path.path_s3 import fetch_files
from impresso_commons.utils.s3 import alternative_read_text, IMPRESSO_STORAGEOPT
from impresso_commons.versioning.helpers import read_manifest_from_s3_path
import dask.bag as db
from tqdm import tqdm
from copy import deepcopy
from datetime import datetime

#### Ensure the environment variables are correctly set.

Note: If you do not have an API key for the BCUL API, ask for them to be shared to you.

In [ ]:
# os.environ['BCUL_API_key'], os.environ['BCUL_username']
API_URI = "https://scriptorium.bcu-lausanne.ch/api"

## Code

#### Get the API session access pwd

In [ ]:
session_headers = {"Content-Type": "application/json"}
session_data = {"username": os.environ["BCUL_username"], "key": os.environ["BCUL_API_key"]}
session_request_uri = "https://scriptorium.bcu-lausanne.ch/api/user/login/api"
session_resp = requests.post(session_request_uri, headers=session_headers, data=json.dumps(session_data))

session_resp_dict = json.loads(session_resp.content)
session = session_resp_dict['session']
role = "ROLE_QUALITY_AUDITOR"

session_resp_dict, session

Response to query:

````JSON
{"id":57,"username":"api@bcul.local","email":"api@bcul.local","name":"API Access","resetPassword":false,"anonymous":false,"locale":"fr","localeDefault":"fr","expire":null,"useTovek":false,"autoLogin":false,"roles":[{"id":2,"name":"Public","role":"ROLE_PUBLIC","count":14,"default":false},{"id":3,"name":"User","role":"ROLE_USER","count":3770,"default":true},{"id":6,"name":"Quality auditor","role":"ROLE_QUALITY_AUDITOR","count":34,"default":false}],"session":"[session key]","mask":{"1":196704,"2":173502456,"4":0,"8":15},"notificationStats":{"all":0,"unread":0}}
````

In [ ]:
# define an general header dict to use for all requests, that already contains the session key
headers = {"Authorization": f"Bearer {session}", "Content-Type": "application/json"}

### Exploring which requests can be done

In the documentation, this is a request example:
```bash
! curl -X "GET" "https://scriptorium.bcu-lausanne.ch/api/user" \
-H f"Authorization: Bearer {session}" \
-H "Content-Type: application/json"
````

We want to express it in python to allow for more custom requests

In [ ]:
headers = {"Authorization": f"Bearer {session}", "Content-Type": "application/json"}
request_uri = "https://scriptorium.bcu-lausanne.ch/api/user"
resp = requests.get(request_uri, headers=headers)

json.loads(resp.content)

#### Harvesting

In [ ]:
! curl -X "GET" "https://demo.mediainfo.com/api/harvest/fields" -H f"Authorization: Bearer {session}" -H "Content-Type: application/json"

#### Initial Request

In [ ]:
session

In [ ]:
headers = {"Authorization": f"Bearer {session}", "Content-Type": "application/json"}

In [ ]:
config_1 = {
    "start": 0,
    "dateSearch": None,
    "layout": "list",
    "filters": [],
    "fields": {},
    "limit": 50,
    "dateCalendar": None,
    "sort": [],
    "dateFilter": None,
    "dateInputs": None
}

In [ ]:
json.dumps(config_1)

In [ ]:
req_uri_1 = "https://demo.mediainfo.com/api/browse"
resp_1 = requests.post(req_uri_1, headers=headers, data=json.dumps(config_1))
resp_1, resp_1.content

In [ ]:
! curl -X "POST" "https://demo.mediainfo.com/api/browse" \
      -H f"Authorization: Bearer {session}" \
      -H 'Content-Type: application/json' \
      -d $'{"favorite": false,"start": 0,"stemming": true,"dateSearch": null,"layout": "list","proximity": 5,"filters": [],"fields": {},"limit": 50,"dateCalendar": null,"sort": [],"dateFilter": null,"dateInputs": null}'


In [ ]:
resp_1_d = json.loads(resp_1.content)

resp_1_d, BeautifulSoup(resp_1_d)

### Collecting an object's metadata

In [ ]:
item_id = '171722' #'46165'
req_uri_2 = f"https://scriptorium.bcu-lausanne.ch/api/item/{item_id}/meta"
resp_2 = requests.get(req_uri_2, headers=headers)

print(f"Status code of response: {resp_2.status_code}")
if resp_2.status_code == 200:
    print("Success, reading response")
    resp_2_d = json.loads(resp_2.content)
else:
    print("Failure, no response")
    resp_2_d = None

resp_2_d

In [ ]:
type(resp_2_d)

### Fetching the Item Ids for all BCUL issues

Geneza's API resquest that provides metadata works with an `Item ID`, where the item is an issue, because there is no title-level organization in the API.

The item ID for each issue is stored in the canonical issue files on S3, inside the `iiif_manifest_uri`. 
As a result, this information needs to be fetched back for all issues of each BCUL title. 

The `Item ID` values fetched from the Canonical will then be used to query the API and collect the metadata, which will be put into a large dict.
Once the dict is formed, it needs to be formatted, and written to disk (probably in JSON format) to be ingested in the MySQL DB.

##### First step: fetch all the Item IDs for BCUL issues

In [ ]:
BCUL_TITLES = [
    "ACI", "AV", "Bombe", "Cancoire", "Castigat", "Charivari", "CharivariCH", "CL", "Croquis", "EM", "esta", "FAM", 
    "FAMDE", "FAN", "FAV1", "Fronde", "GAVi", "Grelot", "Griffe", "Guepe1851", "Guepe1887", 
    "JH", "JV", "JVE", "JY2", "MB", "ME", "MESSAGER", "Moniteur", "NS", "NV", "NV1", "NV2", 
    "OBS", "ouistiti", "PAT", "PDL", "PJ", "PS", "RLA", "TouSuIl", "VVS", "VVS1"
]

s3_bucket_name = '12-canonical-final'
manifest_field = 'iiif_manifest_uri'

In [ ]:
bcul_issue_files, _ = fetch_files(s3_bucket_name, compute=False, newspapers_filter=BCUL_TITLES)

In [ ]:
# define a lambda function extracting the canonical issue ID and corresponding Item ID in Scriptorium's API
extract_item_id = lambda x: (x['id'], x[manifest_field].split('/')[-2])

# Fetch all the ids for BCUL data
issue_ids = bcul_issue_files.map(extract_item_id).compute()

In [ ]:
issue_ids[:10]

##### Define a function that will query the API and fetch a response given the item ID

In [ ]:
def API_metadata_for_item(item_id: str, headers:dict=headers) -> tuple[bool, dict | None]:
    req_uri = f"https://scriptorium.bcu-lausanne.ch/api/item/{item_id}/meta"
    resp = requests.get(req_uri, headers=headers)

    #print(f"Status code of response: {resp.status_code}")
    if resp.status_code == 200:
        #print("Success, reading response")
        return True, json.loads(resp.content)
    else:
        #print("Failure, no response")
        return False, None

In [ ]:
fetched_metadata = {}
error_responses = []

# for all issues in the canonical data, fetch the API from the Metadata and append it if the request was successful
for can_id, item_id in tqdm(issue_ids):
    success, metadata = API_metadata_for_item(item_id)
    title_id = can_id.split('-')[0]
    if success:
        if title_id in fetched_metadata:
            fetched_metadata[title_id].append((item_id, metadata))
        else:
            print(f"Adding new title: {title_id}")
            fetched_metadata[title_id] = [(item_id, metadata)]
    else:
        print(f"Error response for cannonical Id {can_id} - {item_id}")
        error_responses.append((can_id, item_id))

In [ ]:
len(error_responses), error_responses

#### Write fetched information to disk to prevent the need to re-query the API.

In [ ]:
data_dir = "/Users/piconti/impresso/impresso-corpus-metadata/data/api_metadata"
temp_data_path = os.path.join(data_dir, "bcul_fetched_issue_metadata.json")

In [ ]:

#with open(temp_data_path, "w") as f:
#    f.write(json.dumps(fetched_metadata, indent=4))

In [ ]:
# re-reading it 
with open(temp_data_path, 'r') as f_in:
    read_data = json.load(f_in)

read_data

Process the fetched information to aggregate the information per-title

In [ ]:
type(read_data), len(read_data)

In [ ]:
fetched_metadata = None

In [ ]:
bcul_metadata = deepcopy(fetched_metadata) if fetched_metadata else deepcopy(read_data)
len(bcul_metadata)

In [ ]:
key_mappings = {
    'cf14': 'date',
    'cf18': 'editor', # Rédacteur in french, check with Maud!
    'cf17': 'printer', # Imprimeur in french, check with Maud
}

In [ ]:
def add_metadata_to_title(current_data:dict, m_key:str, m_val:dict, key_mappings=key_mappings) -> dict:
    # based on the key, and whether or not it's already in the current data, add new values
    
    # some keys are identified with codes, directly convert them
    m_key = key_mappings[m_key] if m_key in key_mappings else m_key
    # only use the value associated with the key within the m_val dict, which is always a list
    if m_key != '_list':
        vals = m_val['values'] 
    

    # some keys are not desired
    if m_key not in ['_list', 'size_bytes', 'size_pixels']:
        if m_key in current_data:
            for val in vals:
                if m_key in ['id', 'date', 'images']:
                    # the values for these keys change for each item
                    current_data[m_key].append(val)
                elif val not in current_data[m_key]:
                    # the values for other keys are constant per title
                    print(f"    - Found non-matching data for {m_key}, adding the entry")
                    current_data[m_key].append(val)
        else:
            # if key is not present, add the value
            current_data[m_key] = vals
    
    return current_data

In [ ]:
title_metadata = {}

for np_title, issues in bcul_metadata.items():
    current_data = {}
    print(f"––––––– {np_title} –––––––")
    for item_id, item_data in tqdm(issues):
        #print(f"    * item_id: {item_id}, item_data: {item_data}")
        for key, val in item_data.items():
            current_data = add_metadata_to_title(current_data, key, val)
            """if key not in ['_list']:
                print(f"    ** key: {key}, val: {val}")
                if key in current_data:
                    print(f"    *** current_data[key]: {current_data[key]}, \n  val not in current_data[key]: {val not in current_data[key]}, \n    current_data: {current_data}")
                    if val not in current_data[key]:
                        print(f"{np_title}, found non-matching data for {key}, adding the entry")
                        current_data[key].append(val)
                    else:
                        print(f"{np_title}, data for {key} already saved, not adding it")
                else:
                    print(f"{np_title}, adding new key {key}, with value: {[val]}")
                    current_data[key] = [val]"""

    print(f" {np_title}: gathering relevant information from the fetched metadata")
    # gather the relevant information from the list of IDs or dates
    current_data['item_count'] = len(current_data['id'])
    current_data['start_date'] = min(current_data['date'])
    current_data['end_date'] = max(current_data['date'])
    current_data['page_count'] = sum(current_data['images'])
    title_metadata[np_title] = current_data
        
        #f 'cf14' in current_data and item_data['cf14'] != current_data[np_title]['cf14']:
            #print(f"{np_title}, found non-matching data for ")

In [ ]:
title_metadata

#### Comparing the statistics to the ones in the canonical manifest

In [ ]:
s3_path = 's3://12-canonical-final/canonical_v4-5-0.json'

manifest = read_manifest_from_s3_path(s3_path)
manifest

In [ ]:
# list of titles for which the stats don't fit with the metadata
mismatching_titles = {}

for media in tqdm(manifest['media_list']):
    title = media['media_title']
    if title in title_metadata:
        # only considering the BCUL titles
        title_stats = media['media_statistics'][0]
        if title_stats['granularity'] == 'title':
            item_count = title_stats['nps_stats']['issues'] == title_metadata[title]['item_count']
            page_count = title_stats['nps_stats']['pages'] == title_metadata[title]['page_count']
            if item_count and page_count:
                print(f"{title}: stats match with manifest")
            else:
                print(f"{title}: stats DON'T with manifest!!")
                print(f"    issues: {title_stats['nps_stats']['issues']} VS {title_metadata[title]['item_count']}")
                print(f"    pages: {title_stats['nps_stats']['pages']} VS {title_metadata[title]['page_count']}")
                mismatching_titles[title] = {
                    'man_issues': title_stats['nps_stats']['issues'],
                    'man_pages': title_stats['nps_stats']['pages'],
                    'meta_issues': title_metadata[title]['item_count'],
                    'meta_pages': title_metadata[title]['page_count'],
                }
        else:
            print(f"{title}: title_stats don't have the title granularity: {title_stats['granularity']}")

#### Results:

Only TouSuIl has a mismatch of 3 issues - which is known and due to duplicated issues in the files provided by the BCUL. 

### Save the result to json - before its last postprocessing/reformatting

In [ ]:
correct_metadata = deepcopy(title_metadata)

In [ ]:
data_dir = "/Users/piconti/impresso/impresso-corpus-metadata/data/api_metadata"
bcul_data_path = os.path.join(data_dir, "bcul_to_prep_metadata.json")

In [ ]:
with open(bcul_data_path, "w", encoding='utf-8') as f:
    f.write(json.dumps(correct_metadata, indent=4))

In [ ]:
# re-reading it 
with open(bcul_data_path, 'r') as f_in:
    correct_metadata = json.load(f_in)

correct_metadata

First define the string values allowing to classify and extract some info from the description and records.

In [ ]:
html_par = ['<p>', '</p>']
line_break = '<br>'

printer_in_descr = ['Imprimeur: ', 'Imprimeurs : ', 'Imprimeur : ']
red_in_desc = ['Rédacteur: ']
bib_note = ['Note: ']
arch_holder = ['Coll. scannée: ', 'Collection scannée: ', 'Collection scannée : ', 'Collection scannée\xa0: ']
subtitle = ['Le sous-titre varie :']

separators = {
    'printer': printer_in_descr,
    'editor': red_in_desc,
    'note_genealogy': bib_note,
    'archival_holder': arch_holder,
    'subtitle': subtitle
}

In [ ]:
def classify_sep(sep_name, sep_list, section, new_secs):
    sep_match = [p in section for p in sep_list]
    try:
        # there is a match
        index = sep_match.index(1)
        res_section = section.replace(sep_list[index], '')
        if sep_name in new_secs:
            if res_section not in new_secs[sep_name] and res_section+' ' not in new_secs[sep_name]:
                new_secs[sep_name].append(res_section)
                print(f" - updated section: {section}, {sep_list}")
                return True
        else:
            new_secs[sep_name] = [res_section]
            print(f" - updated section: {section}, {sep_list}")
            return True
    except ValueError:
        print(f"No match: {section}, {sep_list}")
        return False

In [ ]:
def process_descr(descr: list[str], separators: dict[str, list|str]=separators):
    result_info = {}
    print(descr)
    # separate all the sections from the description into a flat list, without duplicates
    sections = list(set([x for e in descr for dt in BeautifulSoup(e, 'html') for x in dt.get_text(separator="STOP").split('STOP')]))

    for sec in sections:
        # extract info about each section 
        for sep_name, sep_list in separators.items():
            if classify_sep(sep_name, sep_list, sec, result_info):
                break
    return result_info


In [ ]:
def update_fields_w_decr(prev_meta: dict[str, list|int], descr: dict, key: str) -> tuple[dict, dict]:
    if key in prev_meta and key in descr:
        for v in descr[key]:
            if v not in prev_meta[key]:
                print(f"Extending the existing {key} ({prev_meta[key]}) with new description contents: {v}")
                prev_meta[key].append(v)
        print(f"Removing key {key} from parsed description dict: {descr}")
        del descr[key]
    return prev_meta, descr

In [ ]:
def to_str_date(int_date:int) -> str:
    if int_date % 10000 == 0:
        int_date += 101
        print(f"Reformating date: changed it from {int_date-101} to {int_date}")
    if int_date % 100 == 0:
        int_date += 1
        print(f"Reformating date: changed it from {int_date-1} to {int_date}")
    return str(datetime.date(datetime.strptime(str(int_date), '%Y%m%d')))

In [ ]:
def process_title(meta_titles: list[str], html_par:list[str]=html_par, num_str:str=' Nº'):
    # process the titles fetched. they contain html separators (eg. "<p>""), and sometimes numbers (eg. "N°2")
    new_titles = []
    for t in meta_titles:
        new_t = t.replace(html_par[0], '').replace(html_par[1], '')
        if num_str in new_t:
            new_t = new_t.split(num_str)[0]
        if new_t not in new_titles:
            print(f"Adding new_t ({new_t}) to titles: {new_titles}")
            new_titles.append(new_t)

    return new_titles

In [ ]:
final_bcul_metadata = {}

for title, meta in correct_metadata.items():

    # add title alias to the metadata:
    meta['title_alias'] = title
    # process the descriptions fetched
    descr = process_descr(meta['description'])
    # add potential printers and editors from the description to the ones already fetched
    new_meta, new_descr = update_fields_w_decr(deepcopy(meta), deepcopy(descr), 'printer')
    new_meta, new_descr = update_fields_w_decr(new_meta, new_descr, 'editor')
    # add all other new keys
    new_meta.update(new_descr)
    # reformat the titles to remove the <p> and </p>
    #for t in new_meta['title']:
    #    for par_sign in html_par:
    #        t.replace(par_sign, '')
    new_meta['title'] = process_title(deepcopy(meta['title']))

    # remove the images, dates and ids
    del new_meta['id']
    del new_meta['date']
    del new_meta['images']

    # reformat the dates
    new_meta['start_date'] = str(new_meta['start_date'])[:4]
    new_meta['end_date'] = str(new_meta['end_date'])[:4]
    
    final_bcul_metadata[title] = new_meta

In [ ]:
final_bcul_metadata

#### Save the final metadata

In [ ]:
data_dir = "/Users/piconti/impresso/impresso-corpus-metadata/data/api_metadata"
bcul_final_data_path = os.path.join(data_dir, "api_metadata.bcul.json")

In [ ]:
with open(bcul_final_data_path, "w", encoding='utf-8') as f:
    f.write(json.dumps(list(final_bcul_metadata.values()), indent=4))